In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import pandas as pd
import polars as pl
from datetime import datetime
import requests

from poly_data.analysis.io import scan_markets, scan_trades
from poly_data.analysis.punter import PLATFORM_WALLETS


In [2]:
# Polars display prefs
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(-1)
_ = pl.Config.set_tbl_width_chars(1000)

# Paths
DATA_ROOT = Path("../data")

# Cutoffs / knobs
RECENCY_CUTOFF_STR = "2025-10-01"   # filter traders whose last market activity is after this date
MIN_MARKETS_TRADED = 300            # require enough breadth
BIG_WIN_THRESH = 70.0               # percent change considered a "big win"
MIN_BIG_WINS = 0                    # optionally require some minimum number of big wins to qualify (set >0 to enforce)


In [3]:
markets_df = scan_markets(DATA_ROOT).collect()


In [4]:
# Build a lazy trades frame with timestamp + last_price (per market_id, side).
# Trades parquet is large (multi-GB), so the main frame stays lazy until
# per-wallet filtering shrinks it. The last_price table is small (one row per
# market_id, side), so we materialise it once and broadcast-join it back in.
trades_lf = (
    scan_trades(DATA_ROOT)
    .with_columns(pl.from_epoch("timestamp", time_unit="s").alias("timestamp"))
)

last_price_df = (
    trades_lf
    .sort("timestamp")
    .group_by(["market_id", "nonusdc_side"])
    .agg(pl.col("price").last().alias("last_price"))
    .with_columns(
        pl.when(pl.col("last_price") > 0.98).then(pl.lit(1.0))
          .when(pl.col("last_price") < 0.02).then(pl.lit(0.0))
          .otherwise(pl.col("last_price"))
          .alias("last_price")
    )
    .collect()
)

trades_lf = trades_lf.join(last_price_df.lazy(), on=["market_id", "nonusdc_side"], how="left")


In [5]:
# %% ----------------------- Candidate trader pool -----------------------
# The public Polymarket leaderboard endpoint is not reliably reachable, so we
# derive the analysis cohort directly from the local trades. We then materialise
# only the subset of trades made by that cohort so per-wallet metric calls do
# not re-scan the full multi-GB parquet store every iteration.
TOP_N_TRADERS = 100

plats = list(PLATFORM_WALLETS)

top_traders_df = (
    trades_lf
    .filter(~pl.col("maker").is_in(plats))
    .group_by("maker")
    .agg(
        pl.col("usd_amount").sum().alias("volume_usd"),
        pl.len().alias("fills"),
    )
    .sort("volume_usd", descending=True)
    .head(TOP_N_TRADERS)
    .collect()
)

cohort_wallets = top_traders_df["maker"].to_list()

top_n = (
    top_traders_df
    .rename({"maker": "user_id"})
    .with_columns(pl.col("user_id").alias("user_name"))
    .to_pandas()
)

# Materialise just the cohort's trades. This collapses the multi-GB scan to a
# manageable in-memory frame that get_metrics() can filter cheaply per wallet.
df = (
    trades_lf
    .filter(pl.col("maker").is_in(cohort_wallets))
    .select(
        "timestamp", "market_id", "maker", "taker", "maker_direction",
        "nonusdc_side", "price", "token_amount", "usd_amount",
        "transactionHash", "last_price",
    )
    .collect()
)

print(f"cohort wallets: {len(cohort_wallets)}, fills materialised: {df.height:,}")


cohort wallets: 100, fills materialised: 17,786,989


In [6]:
# %% ----------------------- Per-trader metrics helpers -----------------------
def get_metrics(wallet_address: str) -> pl.DataFrame:
    """
    Build side-level metrics for a single wallet, then compute per-side PnL pieces and VWAPs.
    """
    global df, markets_df

    trader_df = df.filter(pl.col("maker") == wallet_address)
    if trader_df.height == 0:
        return pl.DataFrame(
            schema={
                "market_id": pl.Utf8,
                "side": pl.Utf8,
                "trades": pl.Int64,
                "avg_buy_price": pl.Float64,
                "avg_sold_price_only": pl.Float64,
                "avg_sell_price": pl.Float64,
                "last_price": pl.Float64,
                "total_pnl_usd": pl.Float64,
                "last_trade_ts": pl.Datetime,
                "buy_usd": pl.Float64,
                "sell_usd": pl.Float64,
                "buy_tokens": pl.Float64,
                "sell_tokens": pl.Float64,
                "buy_notional": pl.Float64,
                "sell_notional": pl.Float64,
                "question": pl.Utf8,
            }
        )

    trader_df = trader_df.rename({"maker_direction": "direction", "nonusdc_side": "side"})

    metrics_df = (
        trader_df
        .group_by(["market_id", "side"])
        .agg(
            (pl.when(pl.col("direction") == "BUY").then(pl.col("usd_amount")).otherwise(0.0)).sum().alias("buy_usd"),
            (pl.when(pl.col("direction") == "SELL").then(pl.col("usd_amount")).otherwise(0.0)).sum().alias("sell_usd"),
            (pl.when(pl.col("direction") == "BUY").then(pl.col("token_amount")).otherwise(0.0)).sum().alias("buy_tokens"),
            (pl.when(pl.col("direction") == "SELL").then(pl.col("token_amount")).otherwise(0.0)).sum().alias("sell_tokens"),
            (pl.when(pl.col("direction") == "BUY").then(pl.col("price") * pl.col("token_amount")).otherwise(0.0)).sum().alias("buy_notional"),
            (pl.when(pl.col("direction") == "SELL").then(pl.col("price") * pl.col("token_amount")).otherwise(0.0)).sum().alias("sell_notional"),
            pl.col("timestamp").max().alias("last_trade_ts"),
            pl.len().alias("trades"),
            pl.col("last_price").last().alias("last_price"),
        )
        .with_columns(
            (pl.col("sell_usd") - pl.col("buy_usd")).alias("cash_pnl_usd"),
            (pl.col("buy_tokens") - pl.col("sell_tokens")).alias("inventory_tokens"),
        )
        .with_columns(
            (pl.col("inventory_tokens") * pl.col("last_price")).alias("unrealized_usd"),
        )
        .with_columns(
            (pl.col("cash_pnl_usd") + pl.col("unrealized_usd")).alias("total_pnl_usd"),
        )
        .with_columns(
            pl.when(pl.col("buy_tokens") > 0)
              .then(pl.col("buy_notional") / pl.col("buy_tokens"))
              .otherwise(None)
              .alias("avg_buy_price"),
            pl.when(pl.col("sell_tokens") > 0)
              .then(pl.col("sell_notional") / pl.col("sell_tokens"))
              .otherwise(None)
              .alias("avg_sold_price_only"),
        )
        .with_columns(
            (pl.col("sell_tokens") + pl.col("inventory_tokens")).alias("effective_exit_tokens"),
            (pl.col("sell_notional") + pl.col("inventory_tokens") * pl.col("last_price")).alias("effective_exit_notional"),
        )
        .with_columns(
            pl.when(pl.col("effective_exit_tokens") > 0)
              .then(pl.col("effective_exit_notional") / pl.col("effective_exit_tokens"))
              .otherwise(None)
              .alias("avg_sell_price")
        )
        .select(
            "market_id", "side", "trades",
            "avg_buy_price", "avg_sold_price_only", "avg_sell_price",
            "last_price", "total_pnl_usd",
            "last_trade_ts",
            "buy_usd", "sell_usd",
            "buy_tokens", "sell_tokens",
            "buy_notional", "sell_notional",
        )
        .sort(["market_id", "side"])
    )

    metrics_df = metrics_df.join(
        markets_df.select(["id", "question"]),
        left_on="market_id", right_on="id", how="left"
    )

    metrics_df = metrics_df.with_columns(
        (((pl.col('avg_sell_price') - pl.col('avg_buy_price')) / pl.col('avg_buy_price')) * 100)
        .alias('pct_change')
    )

    metrics_df = metrics_df.drop_nulls(subset=["market_id"])
    return metrics_df


In [7]:
def combine_market_pct(metrics_df: pl.DataFrame) -> pl.DataFrame:
    """
    Collapse (market_id, side) -> (market_id) by taking a buy-USD-weighted average
    of pct_change across sides. Also sum PnL/volumes and keep last_trade_ts (max).
    """
    if metrics_df.height == 0:
        return pl.DataFrame(
            schema={
                "market_id": pl.Utf8,
                "question": pl.Utf8,
                "trades": pl.Int64,
                "pct_change_combined": pl.Float64,
                "total_pnl_usd": pl.Float64,
                "buy_usd_total": pl.Float64,
                "sell_usd_total": pl.Float64,
                "last_trade_ts": pl.Datetime,
            }
        )

    tmp = (
        metrics_df
        .with_columns(
            pl.when(pl.col("pct_change").is_not_null()).then(pl.col("buy_usd")).otherwise(0.0).alias("weight_usd"),
            (pl.col("pct_change") * pl.col("buy_usd")).fill_null(0.0).alias("num_pct_x_usd"),
        )
        .group_by("market_id")
        .agg(
            pl.col("question").first().alias("question"),
            pl.col("trades").sum().alias("trades"),
            pl.col("total_pnl_usd").sum().alias("total_pnl_usd"),
            pl.col("buy_usd").sum().alias("buy_usd_total"),
            pl.col("sell_usd").sum().alias("sell_usd_total"),
            pl.col("num_pct_x_usd").sum().alias("num_pct_x_usd"),
            pl.col("weight_usd").sum().alias("den_usd"),
            pl.col("last_trade_ts").max().alias("last_trade_ts"),
        )
        .with_columns(
            pl.when(pl.col("den_usd") > 0)
              .then(pl.col("num_pct_x_usd") / pl.col("den_usd"))
              .otherwise(None)
              .alias("pct_change_combined")
        )
        .select(
            "market_id", "question", "trades",
            "pct_change_combined",
            "total_pnl_usd", "buy_usd_total", "sell_usd_total",
            "last_trade_ts",
        )
        .sort("market_id")
    )
    return tmp


def _safe_median(series: pl.Series):
    return series.median() if series.len() > 0 else None


def compute_trader_metrics(wallet_address: str) -> dict:
    """
    Robust per-trader stats based on per-market outcomes (one row per market).
    """
    metrics_df = get_metrics(wallet_address)
    c_df = combine_market_pct(metrics_df).rename({"pct_change_combined": "pct_change"})

    n = c_df.height
    if n == 0:
        return {
            "wallet": wallet_address, "markets_traded": 0, "last_trade": None,
            "win_rate": None, "big_win_rate_overall": None, "big_win_rate_among_wins": None,
            "median_win_pct": None, "median_loss_pct": None,
            "median_win_usd": None, "median_loss_usd": None,
            "total_pnl_usd": 0.0, "big_wins_count": 0
        }

    last_trade = c_df["last_trade_ts"].max()
    wins   = c_df.filter(pl.col("total_pnl_usd") > 0)
    losses = c_df.filter(pl.col("total_pnl_usd") <= 0)

    w = wins.height
    l = losses.height

    big_wins = wins.filter(pl.col("pct_change") >= BIG_WIN_THRESH)
    bw = big_wins.height

    win_rate = w / n if n > 0 else None
    big_win_rate_overall = bw / n if n > 0 else None
    big_win_rate_among_wins = (bw / w) if w > 0 else None

    median_win_pct  = _safe_median(wins["pct_change"]) if w > 0 else None
    median_loss_pct = _safe_median(losses["pct_change"]) if l > 0 else None
    median_win_usd  = _safe_median(wins["total_pnl_usd"]) if w > 0 else None
    median_loss_usd = _safe_median(losses["total_pnl_usd"]) if l > 0 else None

    total_pnl_usd   = float(c_df["total_pnl_usd"].sum())

    return {
        "wallet": wallet_address,
        "markets_traded": n,
        "last_trade": last_trade,
        "win_rate": win_rate,
        "big_win_rate_overall": big_win_rate_overall,
        "big_win_rate_among_wins": big_win_rate_among_wins,
        "median_win_pct": median_win_pct,
        "median_loss_pct": median_loss_pct,
        "median_win_usd": median_win_usd,
        "median_loss_usd": median_loss_usd,
        "total_pnl_usd": total_pnl_usd,
        "big_wins_count": bw,
    }


In [8]:
# %% ----------------------- Run over leaderboard & rank -----------------------
rows = []
for _, r in top_n.iterrows():
    stats = compute_trader_metrics(wallet_address=r["user_id"])
    stats["user_name"] = r.get("user_name", None)
    print(stats)

    rows.append(stats)

rank_df = pd.DataFrame(rows)

# Normalize & filter
rank_df["last_trade"] = pd.to_datetime(rank_df["last_trade"])
recency_cutoff = pd.to_datetime(RECENCY_CUTOFF_STR)

# Core filters
rank_df = rank_df[
    (rank_df["markets_traded"] >= MIN_MARKETS_TRADED) &
    (rank_df["last_trade"] > recency_cutoff) &
    (rank_df["win_rate"].notna())
].copy()

# Optional: require at least N big wins
if MIN_BIG_WINS > 0:
    rank_df = rank_df[rank_df["big_wins_count"] >= MIN_BIG_WINS].copy()

# Convenient percent display columns
rank_df["win_rate_pct"] = (rank_df["win_rate"] * 100).round(2)
rank_df["big_win_rate_overall_pct"] = (rank_df["big_win_rate_overall"] * 100).round(2)
rank_df["big_win_rate_among_wins_pct"] = (rank_df["big_win_rate_among_wins"] * 100).round(2)

# Sort to emphasize consistent big winners (then typical size, then $ impact)
rank_df = rank_df.sort_values(
    by=[
        "big_win_rate_among_wins",  # consistency of landing big wins when they do win
        "median_win_pct",           # typical magnitude of wins (pct)
        "total_pnl_usd"             # economic impact
    ],
    ascending=[False, False, False]
)

# Output: Top 50 table
top50_cols = [
    "user_name", "wallet", "markets_traded", "last_trade",
    "win_rate_pct", "big_win_rate_overall_pct", "big_win_rate_among_wins_pct",
    "median_win_pct", "median_loss_pct",
    "median_win_usd", "median_loss_usd",
    "big_wins_count",
    "total_pnl_usd"
]
top50 = rank_df[top50_cols].head(50)
top50.reset_index(drop=True, inplace=True)


{'wallet': '0xd218e474776403a330142299f7796e8ba32eb5c9', 'markets_traded': 562, 'last_trade': datetime.datetime(2026, 4, 13, 1, 32, 1), 'win_rate': 0.7580071174377224, 'big_win_rate_overall': 0.008896797153024912, 'big_win_rate_among_wins': 0.011737089201877934, 'median_win_pct': 1.010101010101011, 'median_loss_pct': -4.647354993907699, 'median_win_usd': 0.7037849999999892, 'median_loss_usd': -3.589999999999999, 'total_pnl_usd': -23391.327222746862, 'big_wins_count': 5, 'user_name': '0xd218e474776403a330142299f7796e8ba32eb5c9'}
{'wallet': '0x5bffcf561bcae83af680ad600cb99f1184d6ffbe', 'markets_traded': 1, 'last_trade': datetime.datetime(2025, 9, 12, 12, 9, 13), 'win_rate': 1.0, 'big_win_rate_overall': 0.0, 'big_win_rate_among_wins': 0.0, 'median_win_pct': 14.157958844061202, 'median_loss_pct': None, 'median_win_usd': 13885.16460974401, 'median_loss_usd': None, 'total_pnl_usd': 13885.16460974401, 'big_wins_count': 0, 'user_name': '0x5bffcf561bcae83af680ad600cb99f1184d6ffbe'}
{'wallet': '